# Vector Embedding

# 00. Initiation

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Projects/hnscc-rag')
print(os.getcwd())

from pathlib import Path
#BASE_DIR = Path(os.getcwd())
#DATA_DIR = BASE_DIR/'data'
#!ls $DATA_DIR

Mounted at /content/drive
/content/drive/MyDrive/Projects/hnscc-rag


In [3]:
# Install Library
!pip install -U sentence-transformers chromadb tqdm -q

In [4]:
# verify library
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm.auto import tqdm
import json

print(f"Torch Version: {torch.__version__}")
if torch.cuda.is_available():
  print(f"GPU Name: {torch.cuda.get_device_name(0)}")
  !nvidia-smi
else:
  print("No GPU")

Torch Version: 2.10.0+cu128
GPU Name: Tesla T4
Fri May  1 09:07:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+

In [5]:
# ============================================================
# Configuration
# ============================================================

DRIVE_BASE = Path("/content/drive/MyDrive/Projects/hnscc-rag")
LOCAL_BASE = Path("/content/hnscc-rag") # working dir on Colab VM

ABSTRACTS_FILE = DRIVE_BASE / "data" / "hnscc_abstracts.json"
CHROMA_DB_PATH = LOCAL_BASE / "chroma_db" # build locally first (faster I/O)
ZIPPED_OUTPUT = DRIVE_BASE / "data" / "chroma_db.zip"

EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
COLLECTION_NAME = "hnscc_abstracts"
BATCH_SIZE = 32

# Create local working directory
LOCAL_BASE.mkdir(parents=True, exist_ok=True)

print(f"Abstracts file: {ABSTRACTS_FILE}")
print(f"Exists: {ABSTRACTS_FILE.exists()}")
print(f"ChromaDB will be built at: {CHROMA_DB_PATH}")
print(f"Final zip will be at: {ZIPPED_OUTPUT}")

Abstracts file: /content/drive/MyDrive/Projects/hnscc-rag/data/hnscc_abstracts.json
Exists: True
ChromaDB will be built at: /content/hnscc-rag/chroma_db
Final zip will be at: /content/drive/MyDrive/Projects/hnscc-rag/data/chroma_db.zip


# 01. Load Abstracts and Embedding Model

In [6]:
# Load abstracts
with ABSTRACTS_FILE.open("r", encoding="utf-8") as f:
 records = json.load(f)

print(f"Loaded {len(records)} abstracts")
print(f"\nSample record keys: {list(records[0].keys())}")
print(f"\nFirst record preview:")
print(f" PMID: {records[0]['pmid']}")
print(f" Title: {records[0]['title'][:100]}...")
print(f" Abstract (first 200 chars): {records[0]['abstract'][:200]}...")

Loaded 3000 abstracts

Sample record keys: ['pmid', 'title', 'abstract', 'authors', 'journal', 'year', 'publication_types', 'mesh_terms', 'doi', 'url', 'source_queries']

First record preview:
 PMID: 42031720
 Title: Mutational signature-based classification uncovers emerging oral cancer subtypes with distinct molec...
 Abstract (first 200 chars): Tobacco use, alcohol consumption, and infection with human papilloma virus (HPV) are well-established risk factors for head and neck squamous cell carcinomas (HNSCC). However, the incidence of oral ca...


In [7]:
# Load Embedding mdoel
print(f"Loading {EMBEDDING_MODEL} ...")

model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")
print(f"\nModel Loaded")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"  Mac Sequence length: {model.max_seq_length}")
print(f"  Device: {model.device}")


Loading pritamdeka/S-PubMedBert-MS-MARCO ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model Loaded
  Embedding dimension: 768
  Mac Sequence length: 350
  Device: cuda:0


/tmp/ipykernel_8476/3057476663.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")


# 02. Text for Embeddings

In [8]:
list(records[0].keys())

['pmid',
 'title',
 'abstract',
 'authors',
 'journal',
 'year',
 'publication_types',
 'mesh_terms',
 'doi',
 'url',
 'source_queries']

In [9]:
texts = []
metadatas = []
ids = []

for r in records:
  combined_text = f"{r['title']} {r['abstract']}"
  texts.append(combined_text)

  metadata = {
      "pmid": r["pmid"],
      "title": r["title"],
      "year": r.get("year", 0),
      "journal": r.get("journal", ""),
      "doi": r.get("doi", ""),
      "url": r.get("url", ""),
      "source_queries": ",".join(r.get("source_queries", [])),
      "abstract": r["abstract"][:8000]
  }
  metadatas.append(metadata)
  ids.append(r["pmid"])


print(f"Prepared {len(texts)} texts for embedding")
print(f"Sample text length (chars): {len(texts[0])}")
print(f"Average text length (chars): {sum(len(t) for t in texts) / len(texts):.0f}")

Prepared 3000 texts for embedding
Sample text length (chars): 2137
Average text length (chars): 1789


# 03. Generate Embeddings

In [10]:
import time
print(f"Encoding {len(texts)}")
print(f"Batch size: {BATCH_SIZE}\n")

start = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True #L2 Normalize for cosine similarity
)

elapsed = time.time() - start

print(f"\nEncoding Complete")
print(f"  Time elapsed: {elapsed:.1f} seconds ({elapsed/60:.1f} min)")
print(f"  Embeddings shape: {embeddings.shape}")
print(f"  Embeddings dtype: {embeddings.dtype}")

Encoding 3000
Batch size: 32



Batches:   0%|          | 0/94 [00:00<?, ?it/s]


Encoding Complete
  Time elapsed: 58.5 seconds (1.0 min)
  Embeddings shape: (3000, 768)
  Embeddings dtype: float32


# 04. Index into ChromaDB

In [11]:
import shutil

# Clean any previous chromaDB
if CHROMA_DB_PATH.exists():
  shutil.rmtree(CHROMA_DB_PATH)

# create persistent ChromaDB client
client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

# Create collection with cosine similarity
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"} # cosine similarity
)

print(f"Created collection: {COLLECTION_NAME}")
print(f"Path: {CHROMA_DB_PATH}")

Created collection: hnscc_abstracts
Path: /content/hnscc-rag/chroma_db


In [12]:
# Bulk Insert
# ChromaDB handles batches natively but very large inserts can hit memory limits
# Insert in chunks of 500 to be safe

CHUNK_SIZE = 500

for i in tqdm(range(0, len(ids), CHUNK_SIZE), desc="Indexing"):
  chunk_end = min(i + CHUNK_SIZE, len(ids))
  collection.add(
    ids=ids[i:chunk_end],
    embeddings=embeddings[i:chunk_end].tolist(),
    documents=texts[i:chunk_end],
    metadatas=metadatas[i:chunk_end],
  )

print(f"\nIndexed {collection.count()} documents")

Indexing:   0%|          | 0/6 [00:00<?, ?it/s]


Indexed 3000 documents


In [20]:
# Sanity check retrieval
# Test 5 queries
test_queries = [
 "HPV positive oropharyngeal cancer prognosis",
 "PD-L1 immunotherapy head and neck cancer",
 "tumor microenvironment immune infiltration HNSCC",
 "EGFR targeted therapy laryngeal cancer",
 "DNA methylation biomarker oral squamous cell carcinoma",
]
for q in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: {q}")
    print(f"{'='*70}")

    # Encode query
    query_emb = model.encode([q], normalize_embeddings=True)[0].tolist()

    # Retrieve top 3
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=3,
    )

    # 🔥 FIX: this loop must be inside
    for i, (doc_id, distance, metadata) in enumerate(zip(
        results["ids"][0],
        results["distances"][0],
        results["metadatas"][0]
    )):
        # Cosine distance → similarity
        similarity = 1 - distance

        print(f"\n[{i+1}] PMID: {metadata['pmid']} | Similarity: {similarity:.3f} | Year: {metadata['year']}")
        print(f" {metadata['title'][:120]}...")


Query: HPV positive oropharyngeal cancer prognosis

[1] PMID: 40960014 | Similarity: 0.951 | Year: 2026
 Clinical Outcomes of Radiologically Node-Negative Oropharyngeal Squamous Cell Carcinoma: A Retrospective Analysis....

[2] PMID: 39291666 | Similarity: 0.951 | Year: 2025
 Prognostic Significance of Human Papillomavirus Genotypes in Oropharyngeal Squamous Cell Carcinoma....

[3] PMID: 39397259 | Similarity: 0.949 | Year: 2024
 Differential molecular characterization of human papillomavirus-associated oropharyngeal squamous cell carcinoma and its...

Query: PD-L1 immunotherapy head and neck cancer

[1] PMID: 37207445 | Similarity: 0.955 | Year: 2023
 The current advances and future directions of PD-1/PD-L1 blockade in head and neck squamous cell carcinoma (HNSCC) in th...

[2] PMID: 39370694 | Similarity: 0.953 | Year: 2024
 A Review of Immunotherapy for Head and Neck Cancer....

[3] PMID: 36499710 | Similarity: 0.952 | Year: 2022
 Programmed Cell Death-Ligand 1 in Head and Neck Squ

In [14]:
# Upload To Drive
import shutil

print("Compressing chroma_db folder...")

# Remove old zip if exists
if ZIPPED_OUTPUT.exists():
    ZIPPED_OUTPUT.unlink()

# Create zip (this also takes ~30s)
shutil.make_archive(
    base_name=str(ZIPPED_OUTPUT.with_suffix("")), # without .zip
    format="zip",
    root_dir=str(CHROMA_DB_PATH.parent),
    base_dir=CHROMA_DB_PATH.name,
)

print(f"\nZipped to: {ZIPPED_OUTPUT}")
print(f" Size: {ZIPPED_OUTPUT.stat().st_size / 1024 / 1024:.2f} MB")

Compressing chroma_db folder...

Zipped to: /content/drive/MyDrive/Projects/hnscc-rag/data/chroma_db.zip
 Size: 40.11 MB


In [17]:
!ls -lh $DRIVE_BASE/data

total 63M
-rw------- 1 root root  41M May  1 09:10 chroma_db.zip
-rw------- 1 root root 7.8M Apr 30 11:09 hnscc_abstracts.json
-rw------- 1 root root  15M Apr 30 10:42 hnscc_abstracts_raw.json
